In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'Pi_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.025
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = True
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)



Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: OutGap
Augmented coefficient: Pi_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[0.302 0.    0.   ]
 [0.    0.42  0.   ]
 [0.    0.    0.019]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,16.333,0.003,0.021,0.0,100000,98894,0.989,0.000,0.988,0.990
1,Infl,24.800,0.000,0.025,0.0,100000,99981,1.000,0.000,1.000,1.000
2,Rate,9.926,0.028,0.018,0.0,100000,87156,0.872,0.001,0.869,0.874


In [8]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.121,2.536,0.573,0.000,0.008,0.001,100000,4070,0.041,0.001,0.039,0.042,3.0,200,4
1,cov_identity,15.144,435.471,0.000,0.006,0.476,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-2.094,-0.087,1.716,-1.235,0.310,0.012,0.005,0.0,0.001,0.003,0.001,0.0,100000,22396,0.224,0.001,0.221,0.227
1,OutGap,x,-0.303,-0.097,0.216,-1.384,0.275,0.014,0.001,0.0,0.000,0.003,0.001,0.0,100000,27006,0.270,0.001,0.267,0.273
2,OutGap,r,-0.583,-0.019,2.023,-0.271,0.501,0.005,0.006,0.0,0.001,0.003,0.001,0.0,100000,4879,0.049,0.001,0.047,0.050
3,Infl,Pi,-0.661,-0.022,1.968,-0.316,0.481,0.006,0.006,0.0,0.001,0.003,0.001,0.0,100000,6318,0.063,0.001,0.062,0.065
4,Infl,x,0.051,0.015,0.248,0.209,0.489,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5793,0.058,0.001,0.056,0.059
5,Infl,r,-0.467,-0.013,2.313,-0.182,0.491,0.005,0.008,0.0,0.001,0.003,0.001,0.0,100000,5682,0.057,0.001,0.055,0.058
6,Rate,Pi,-0.111,-0.022,0.370,-0.314,0.487,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5985,0.060,0.001,0.058,0.061
7,Rate,x,0.017,0.026,0.047,0.366,0.480,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,6493,0.065,0.001,0.063,0.066
8,Rate,r,-0.308,-0.048,0.434,-0.685,0.433,0.007,0.001,0.0,0.000,0.003,0.001,0.0,100000,10375,0.104,0.001,0.102,0.106


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-4.064,-0.327,0.829,-4.889,0.000,0.110,0.003,0.0,0.000,0.003,0.000,0.0,100000,99970,1.000,0.000,1.000,1.000
1,OutGap,x,-0.501,-0.327,0.102,-4.896,0.000,0.110,0.000,0.0,0.000,0.003,0.000,0.0,100000,99956,1.000,0.000,0.999,1.000
0,OutGap,r,1.455,0.053,1.910,0.748,0.466,0.005,0.004,0.0,0.001,0.002,0.001,0.0,100000,4150,0.042,0.001,0.040,0.043
5,Infl,Pi,-0.156,-0.010,1.002,-0.136,0.499,0.005,0.003,0.0,0.000,0.003,0.001,0.0,100000,5103,0.051,0.001,0.050,0.052
4,Infl,x,-0.001,0.001,0.123,0.012,0.503,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,4730,0.047,0.001,0.046,0.049
3,Infl,r,-0.414,-0.013,2.184,-0.180,0.495,0.005,0.007,0.0,0.001,0.003,0.001,0.0,100000,5317,0.053,0.001,0.052,0.055
8,Rate,Pi,0.009,0.002,0.188,0.032,0.506,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,4579,0.046,0.001,0.045,0.047
7,Rate,x,0.008,0.022,0.023,0.317,0.490,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,5786,0.058,0.001,0.056,0.059
6,Rate,r,-0.337,-0.056,0.410,-0.791,0.412,0.008,0.001,0.0,0.000,0.003,0.001,0.0,100000,12147,0.121,0.001,0.119,0.124


In [11]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.704541,-3.798249,-2.093708,-2.093708,2.013258e-19,5.247899e-16,1.760122e-15,0.003254,0.002609,0.005224,0.005224,2.175665e-18,1.406928e-18,8.616809e-19
1,OutGap,x,0.020946,-0.323879,-0.302933,-0.302933,-3.375887e-21,5.735899e-17,1.760122e-15,0.000410,0.000357,0.000684,0.000684,2.463611e-19,1.667122e-19,8.616809e-19
2,OutGap,r,-0.236837,-0.345761,-0.582598,-0.582598,7.915518e-19,4.402567e-16,1.760122e-15,0.003861,0.003091,0.006231,0.006231,1.843893e-18,1.208995e-18,8.616809e-19
3,Infl,Pi,-0.026777,-0.633885,-0.660662,-0.660662,-1.816215e-17,4.786725e-16,1.760122e-15,0.001022,0.006280,0.006348,0.006348,1.991848e-18,1.295940e-18,8.616809e-19
4,Infl,x,0.003010,0.048330,0.051340,0.051340,1.373346e-19,5.970087e-17,1.760122e-15,0.000129,0.000793,0.000802,0.000802,2.484368e-19,1.614892e-19,8.616809e-19
5,Infl,r,-0.010624,-0.456162,-0.466786,-0.466786,2.177715e-18,5.562185e-16,1.760122e-15,0.001205,0.007452,0.007526,0.007526,2.322835e-18,1.517165e-18,8.616809e-19
6,Rate,Pi,0.000583,-0.111087,-0.110504,-0.110504,-4.754985e-19,1.752044e-16,1.760122e-15,0.000220,0.001157,0.001164,0.001164,7.023001e-19,4.315773e-19,8.616809e-19
7,Rate,x,-0.000124,0.017260,0.017136,0.017136,5.761762e-20,2.210346e-17,1.760122e-15,0.000028,0.000149,0.000149,0.000149,8.885491e-20,5.485935e-20,8.616809e-19
8,Rate,r,-0.004500,-0.303111,-0.307611,-0.307611,-4.486776e-19,2.151339e-16,1.760122e-15,0.000258,0.001414,0.001413,0.001413,8.654453e-19,5.349460e-19,8.616809e-19


In [12]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.923165e+00,-5.986766,-4.063601,-4.063601,1.588729e-18,6.020373e-16,1.760122e-15,0.001625,0.001144,0.002573,0.002573,2.553535e-18,1.701775e-18,8.616809e-19
1,OutGap,x,2.083532e-01,-0.709834,-0.501481,-0.501481,1.249001e-19,7.286588e-17,1.760122e-15,0.000203,0.000195,0.000336,0.000336,3.100970e-19,2.075221e-19,8.616809e-19
2,OutGap,r,-8.229790e-01,2.277518,1.454539,1.454539,2.773267e-18,4.642599e-16,1.760122e-15,0.004541,0.003552,0.004416,0.004416,1.937667e-18,1.264609e-18,8.616809e-19
3,Infl,Pi,-2.734020e-03,-0.153025,-0.155759,-0.155759,-1.853078e-17,2.420499e-16,1.760122e-15,0.000520,0.003126,0.003160,0.003160,9.969823e-19,6.414987e-19,8.616809e-19
4,Infl,x,1.533367e-04,-0.001565,-0.001412,-0.001412,-2.041869e-18,2.965801e-17,1.760122e-15,0.000064,0.000383,0.000387,0.000387,1.225521e-19,7.914955e-20,8.616809e-19
5,Infl,r,-1.111259e-02,-0.402980,-0.414093,-0.414093,8.054022e-18,5.225194e-16,1.760122e-15,0.001133,0.006894,0.006955,0.006955,2.173646e-18,1.412477e-18,8.616809e-19
6,Rate,Pi,-1.346366e-05,0.009490,0.009477,0.009477,5.754451e-19,8.919327e-17,1.760122e-15,0.000112,0.000579,0.000585,0.000585,3.561499e-19,2.174647e-19,8.616809e-19
7,Rate,x,-7.163557e-09,0.007866,0.007866,0.007866,4.820098e-20,1.100089e-17,1.760122e-15,0.000014,0.000072,0.000073,0.000073,4.399619e-20,2.693473e-20,8.616809e-19
8,Rate,r,-2.956708e-03,-0.333928,-0.336885,-0.336885,-1.752485e-19,2.047751e-16,1.760122e-15,0.000243,0.001337,0.001338,0.001338,8.238915e-19,5.093770e-19,8.616809e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [13]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
        summary_only=_MC_SUMMARY_ONLY,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [14]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.901,-2311.509,-1087.767,2447.484,0.0,0.0,0.574,0.131,1.073,0.0,100000,100000,1.0,0.0,1.0,1.0


In [15]:
res_mle

OptimizationResult(kind='mle', x=array([1.78079954]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(1.7807995439289146), 'x_coef': np.float64(0.0), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(1024.4619336799406), loglik=np.float64(-1024.4619336799406), logprior=np.float64(0.0), logpost=np.float64(-1024.4619336799406), nfev=8, nit=3, raw=  message: CONVERGENCE:

## Serial Autocorrelation Tests for the Augmented Model

In [16]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.329,0.447,0.006,0.001,100000,8654,0.087,0.001,0.085,0.088
1,Infl,24.417,0.000,0.026,0.000,100000,99947,0.999,0.000,0.999,1.000
2,Rate,5.878,0.105,0.014,0.001,100000,61227,0.612,0.002,0.609,0.615


In [17]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.121,2.536,0.573,0.000,0.008,0.001,100000,4070,0.041,0.001,0.039,0.042,3.0,200,4
1,cov_identity,15.144,435.471,0.000,0.006,0.476,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.114,2.830,0.525,0.000,0.008,0.001,100000,4546,0.045,0.001,0.044,0.047,3.0,200,4
1,cov_identity,2.407,63.691,0.000,0.001,0.049,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
